# Download microsoft/bitnet-b1.58-2B-4T

In [1]:
!pip install -U huggingface_hub

  Using cached huggingface_hub-1.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached huggingface_hub-1.6.0-py3-none-any.whl (612 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Using cached anyio-4.12.1-py3-none-any.whl (113 kB)
Using cached annotated_doc-0.0.4-py3-none-any.whl (5.3 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
  Attempting uninstall: huggingface_

In [2]:
from huggingface_hub import snapshot_download

repo_id = "microsoft/BitNet-b1.58-2B-4T-gguf"
local_dir = "./models/BitNet-b1.58-2B-4T-gguf"

snapshot_download(
    repo_id=repo_id,
    local_dir=local_dir,
    local_dir_use_symlinks=False,
    allow_patterns=["*.gguf", "*.md", "*.json", "LICENSE*"],
)

print(f"Downloaded to: {local_dir}")

/home/mr_cobot/anaconda3/envs/bitnet-cpp/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mr_cobot/anaconda3/envs/bitnet-cpp/lib/python3.9/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching 3 files: 100%|██████████| 3/3 [11:18<00:00, 226.17s/it]

Downloaded to: ./models/BitNet-b1.58-2B-4T-gguf


## Model Layer Inspection (GGUF)

In [1]:
from gguf import GGUFReader
from pathlib import Path
from collections import Counter

model_path = "./models/BitNet-b1.58-2B-4T-gguf/ggml-model-i2_s.gguf"
reader = GGUFReader(model_path)

# --- Model metadata ---
print("=" * 60)
print("MODEL METADATA")
print("=" * 60)
for field_name in ["general.architecture", "general.name", "general.quantization_version",
                    "llama.block_count", "llama.context_length", "llama.embedding_length",
                    "llama.attention.head_count", "llama.attention.head_count_kv",
                    "llama.feed_forward_length", "llama.rope.freq_base",
                    "llama.vocab_size"]:
    field = reader.fields.get(field_name)
    if field:
        # Extract value from field parts
        val = field.parts[field.data[0]] if len(field.data) == 1 else [field.parts[i] for i in field.data]
        print(f"  {field_name}: {val}")

# --- Tensor (layer) summary ---
print("\n" + "=" * 60)
print(f"TENSORS / LAYERS  (total: {len(reader.tensors)})")
print("=" * 60)

type_counter = Counter()
print(f"{'Name':<55} {'Shape':<25} {'Type':<12} {'Size (MB)':>10}")
print("-" * 105)
for t in reader.tensors:
    shape = " × ".join(str(d) for d in t.shape)
    size_mb = t.n_bytes / (1024 * 1024)
    type_name = str(t.tensor_type).split(".")[-1]
    type_counter[type_name] += 1
    print(f"  {t.name:<53} {shape:<25} {type_name:<12} {size_mb:>9.2f}")

# --- Summary ---
total_mb = sum(t.n_bytes for t in reader.tensors) / (1024 * 1024)
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  Total tensors : {len(reader.tensors)}")
print(f"  Total size    : {total_mb:.2f} MB")
print(f"  Quant types   : {dict(type_counter)}")

ValueError: 36 is not a valid GGMLQuantizationType